# Three-type push policy — TimesFM window closing

Этот этап переиспользует Stage 22 и меняет только генерацию `window_closing_candidate`. `GOOD_DAY` и `POSITIVE_MARKET_FACT` остаются без изменений. Здесь выполняется только выбор TimesFM-rule на validation 2025; 2026 не читается при выборе и не оценивается.

## Methodology and leakage boundary

Golden methodology задаёт historical eligibility: текущий курс строго более чем на 100, но не более чем на 200 bps выше минимума предыдущих 10 календарных дней. Retrospective label дополнительно требует `good=0` и actual future median deterioration ≥100 bps. Эти future-derived условия используются только как ground truth. В causal candidate входят historical band и TimesFM forecast, но не `good`, `closing`, actual future или другие golden labels. Поскольку Stage 24 обучал обе модели на полном calendar-time ряду, H1…H10 здесь означают следующие 10 календарных дней.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.three_type_push_policy import load_inputs, build_candidates, _binary_metrics
golden, chronos=load_inputs(ROOT)
base=build_candidates(golden,chronos)
base=base[base.date.dt.year.eq(2025)].copy()
raw=pd.read_parquet(ROOT/'reports/window_closing_model_predictions.parquet')
raw['date']=pd.to_datetime(raw.date)
pred_cols=[f'predicted_rate_h{i}' for i in range(1,11)]
# Security boundary: discard every evaluation/future-actual field before merge.
tf=raw.loc[raw.model.eq('TIMESFM'),['corridor','date','rate_t',*pred_cols]].copy()
tf=tf.rename(columns={c:f'timesfm_{c}' for c in pred_cols})
tf=tf[tf.date.dt.year.eq(2025)]
assert not tf.duplicated(['corridor','date']).any()
validation=base.merge(tf,on=['corridor','date','rate_t'],validate='one_to_one')
assert len(validation)==len(base)
print('validation rows:',len(validation),'corridors:',sorted(validation.corridor.unique()))

validation rows: 1305 corridors: ['AMD_RUB', 'KGS_RUB', 'KZT_RUB', 'TJS_RUB', 'UZS_RUB']


In [2]:
timesfm_pred_cols=[f'timesfm_{c}' for c in pred_cols]
p=validation[timesfm_pred_cols].to_numpy(float)
r=validation.rate_t.to_numpy(float)
changes=(p/r[:,None]-1)*10000
validation['change_h3_bps']=changes[:,2]
validation['change_h5_bps']=changes[:,4]
validation['change_h10_bps']=changes[:,9]
validation['mean_change_h3_bps']=changes[:,:3].mean(axis=1)
validation['mean_change_h5_bps']=changes[:,:5].mean(axis=1)
validation['max_deterioration_h10_bps']=changes.max(axis=1)
h5=np.arange(1,6)
validation['slope_h5']=((changes[:,:5]-changes[:,:5].mean(axis=1,keepdims=True))*(h5-h5.mean())).sum(axis=1)/((h5-h5.mean())**2).sum()
validation['n_worse_days_h5']=(changes[:,:5]>0).sum(axis=1)
validation['n_worse_days_h10']=(changes>0).sum(axis=1)
validation['methodology_historical_eligible']=validation.past_rebound_bps.gt(100)&validation.past_rebound_bps.le(200)
score_cols=['change_h3_bps','change_h5_bps','change_h10_bps','mean_change_h3_bps','mean_change_h5_bps','max_deterioration_h10_bps','slope_h5','n_worse_days_h5','n_worse_days_h10']
display(validation[score_cols].describe())

,change_h3_bps,change_h5_bps,change_h10_bps,mean_change_h3_bps,mean_change_h5_bps,max_deterioration_h10_bps,slope_h5,n_worse_days_h5,n_worse_days_h10
count,1305.000000,1305.000000,1305.000000,1305.000000,1305.000000,1305.000000,1305.000000,1305.000000,1305.000000
mean,10.398448,10.880453,21.250574,7.767798,8.772254,38.079381,1.466862,2.922605,5.863602
std,31.755572,42.374278,70.610831,25.139840,30.489942,56.453609,7.384772,2.115743,4.272156
min,-88.668458,-135.670826,-192.996548,-64.188212,-82.130888,-47.186565,-28.797933,0.000000,0.000000
25%,-8.551280,-14.410129,-21.520509,-6.687667,-9.634523,1.206288,-2.812731,0.000000,1.000000
50%,6.508789,4.799377,14.292877,4.626288,4.498259,22.535209,0.846503,4.000000,8.000000
75%,24.639715,29.519717,51.314834,18.342383,22.148204,56.662732,4.797167,5.000000,10.000000
max,203.415406,264.124246,415.529376,169.522519,203.532377,415.529376,47.304947,5.000000,10.000000


## Unchanged signal types

Следующая проверка фиксирует, что Stage 25 не пересчитывает и не изменяет `good_day_candidate` или `positive_market_fact_candidate`. Они напрямую унаследованы из результата Stage 22.

In [3]:
assert validation.good_day_candidate.equals(base.good_day_candidate.reset_index(drop=True))
assert validation.positive_market_fact_candidate.equals(base.positive_market_fact_candidate.reset_index(drop=True))
eligible=validation.methodology_historical_eligible
rules=[]
for threshold in [25,50,75]:
    rules.append((f'A_MEAN_H5_{threshold}',2,eligible&validation.mean_change_h5_bps.ge(threshold)&validation.n_worse_days_h5.ge(3),f'mean_change_h5_bps>={threshold} AND n_worse_days_h5>=3'))
for threshold in [50,75,100]:
    rules.append((f'B_SLOPE_H5_{threshold}',3,eligible&validation.slope_h5.gt(0)&validation.n_worse_days_h5.ge(4)&validation.max_deterioration_h10_bps.ge(threshold),f'slope_h5>0 AND n_worse_days_h5>=4 AND max_deterioration_h10_bps>={threshold}'))
for threshold in [50,75,100]:
    rules.append((f'C_MEAN_H3_MAX_{threshold}',2,eligible&validation.mean_change_h3_bps.gt(0)&validation.max_deterioration_h10_bps.ge(threshold),f'mean_change_h3_bps>0 AND max_deterioration_h10_bps>={threshold}'))
for n in [6,7]:
    for threshold in [50,75]:
        rules.append((f'D_COUNT_{n}_H10_{threshold}',2,eligible&validation.n_worse_days_h10.ge(n)&validation.change_h10_bps.ge(threshold),f'n_worse_days_h10>={n} AND change_h10_bps>={threshold}'))
rows=[]
for rule_id,complexity,signal,description in rules:
    m=_binary_metrics(validation.golden_window_closing,signal)
    rows.append({'rule_id':rule_id,'complexity':complexity,'rule_description':description,'signals':m['signals'],'TP':m['TP'],'FP':m['FP'],'FN':m['FN'],'precision':m['precision'],'recall':m['recall'],'F0_5':m['F0_5'],'F1':m['F1']})
results=pd.DataFrame(rows).sort_values(['F0_5','precision','recall','complexity','rule_id'],ascending=[False,False,False,True,True]).reset_index(drop=True)
display(results)

,rule_id,complexity,rule_description,signals,TP,FP,FN,precision,recall,F0_5,F1
0,A_MEAN_H5_25,2,mean_change_h5_bps>=25 AND n_worse_days_h5>=3,64,25,39,25,0.390625,0.50,0.408497,0.438596
1,B_SLOPE_H5_50,3,slope_h5>0 AND n_worse_days_h5>=4 AND max_dete...,79,28,51,22,0.354430,0.56,0.382514,0.434109
2,A_MEAN_H5_75,2,mean_change_h5_bps>=75 AND n_worse_days_h5>=3,8,6,2,44,0.750000,0.12,0.365854,0.206897
3,D_COUNT_6_H10_50,2,n_worse_days_h10>=6 AND change_h10_bps>=50,81,27,54,23,0.333333,0.54,0.360963,0.412214
4,D_COUNT_7_H10_50,2,n_worse_days_h10>=7 AND change_h10_bps>=50,81,27,54,23,0.333333,0.54,0.360963,0.412214
5,C_MEAN_H3_MAX_50,2,mean_change_h3_bps>0 AND max_deterioration_h10...,82,27,55,23,0.329268,0.54,0.357143,0.409091
6,A_MEAN_H5_50,2,mean_change_h5_bps>=50 AND n_worse_days_h5>=3,23,9,14,41,0.391304,0.18,0.316901,0.246575
7,D_COUNT_6_H10_75,2,n_worse_days_h10>=6 AND change_h10_bps>=75,55,17,38,33,0.309091,0.34,0.314815,0.323810
8,D_COUNT_7_H10_75,2,n_worse_days_h10>=7 AND change_h10_bps>=75,55,17,38,33,0.309091,0.34,0.314815,0.323810
9,B_SLOPE_H5_75,3,slope_h5>0 AND n_worse_days_h5>=4 AND max_dete...,54,16,38,34,0.296296,0.32,0.300752,0.307692


In [4]:
winner=results.iloc[0]
print('SELECTED TIMESFM WINDOW_CLOSING RULE (2025 ONLY)')
display(winner.to_frame().T)
print('rule:',winner.rule_description)
print('validation metrics:',{k:winner[k] for k in ['signals','precision','recall','F0_5','F1']})
print('2026 USED FOR SELECTION: NO')
print('GOOD_DAY UNCHANGED: PASS')
print('POSITIVE_MARKET_FACT UNCHANGED: PASS')

SELECTED TIMESFM WINDOW_CLOSING RULE (2025 ONLY)


,rule_id,complexity,rule_description,signals,TP,FP,FN,precision,recall,F0_5,F1
0,A_MEAN_H5_25,2,mean_change_h5_bps>=25 AND n_worse_days_h5>=3,64,25,39,25,0.390625,0.5,0.408497,0.438596


rule: mean_change_h5_bps>=25 AND n_worse_days_h5>=3
validation metrics: {'signals': np.int64(64), 'precision': np.float64(0.390625), 'recall': np.float64(0.5), 'F0_5': np.float64(0.4084967320261438), 'F1': np.float64(0.43859649122807015)}
2026 USED FOR SELECTION: NO
GOOD_DAY UNCHANGED: PASS
POSITIVE_MARKET_FACT UNCHANGED: PASS


## Frozen three-type policy assembly

Ниже меняется только `window_closing_candidate`: применяется выбранное на validation 2025 правило TimesFM `A_MEAN_H5_25`. `good_day_candidate`, `positive_market_fact_candidate`, приоритет, cooldown, weekly cap и решение по 1-day deferral полностью наследуются из Stage 22. В Stage 22 adaptive ranking зафиксирован как `none`, поэтому новый ranking здесь не вводится. Golden labels и actual future не участвуют в сборке policy.

In [5]:
from src.three_type_push_policy import apply_policy

assert winner.rule_id == 'A_MEAN_H5_25', 'Frozen winner differs from the selected validation rule'
policy_base = build_candidates(golden, chronos)
tf_policy = raw.loc[raw.model.eq('TIMESFM'), ['corridor', 'date', 'rate_t', *pred_cols]].copy()
tf_policy = tf_policy.rename(columns={c: f'timesfm_{c}' for c in pred_cols})
assert not tf_policy.duplicated(['corridor', 'date']).any()
policy_input = policy_base.merge(tf_policy, on=['corridor', 'date', 'rate_t'], how='inner', validate='one_to_one')
assert len(policy_input) == len(policy_base), 'TimesFM coverage does not match the Stage 22 policy rows'

policy_pred = policy_input[timesfm_pred_cols].to_numpy(float)
policy_rate = policy_input.rate_t.to_numpy(float)
policy_changes = (policy_pred / policy_rate[:, None] - 1) * 10_000
policy_input['mean_change_h5_bps'] = policy_changes[:, :5].mean(axis=1)
policy_input['n_worse_days_h5'] = (policy_changes[:, :5] > 0).sum(axis=1)
policy_input['methodology_historical_eligible'] = policy_input.past_rebound_bps.gt(100) & policy_input.past_rebound_bps.le(200)
policy_input['window_closing_candidate_timesfm'] = (
    policy_input.methodology_historical_eligible
    & policy_input.mean_change_h5_bps.ge(25)
    & policy_input.n_worse_days_h5.ge(3)
)
policy_input['window_closing_candidate'] = policy_input.window_closing_candidate_timesfm

# Frozen Stage 22 communication policy: no retuning. The optional deferral remains
# forecast-only, but is disabled because defer_fact=False was frozen on validation 2025.
policy = apply_policy(policy_input, cooldown_days=4, defer_fact=False)
policy['deferred_1d'] = policy.deferred_for_next_day_high_priority

output_columns = [
    'corridor', 'date', 'rate_t',
    'good_day_candidate', 'window_closing_candidate', 'positive_market_fact_candidate',
    'suppressed_by_priority', 'suppressed_by_cooldown', 'deferred_1d',
    'final_push', 'final_push_type', 'final_push_reason',
]
policy_output = policy[output_columns].copy()
output_path = ROOT / 'reports/three_type_push_policy_timesfm.parquet'
policy_output.to_parquet(output_path, index=False)
print('saved:', output_path.relative_to(ROOT))
print('rows:', len(policy_output), 'date range:', policy_output.date.min().date(), '—', policy_output.date.max().date())
display(policy_output.head())

saved: reports\three_type_push_policy_timesfm.parquet
rows: 2145 date range: 2025-01-01 — 2026-08-24


,corridor,date,rate_t,good_day_candidate,window_closing_candidate,positive_market_fact_candidate,suppressed_by_priority,suppressed_by_cooldown,deferred_1d,final_push,final_push_type,final_push_reason
0,AMD_RUB,2025-01-01,0.256456,True,False,False,False,False,False,True,good_day,GOOD_DAY
1,AMD_RUB,2025-01-02,0.256456,True,False,False,False,True,False,False,none,COOLDOWN
2,AMD_RUB,2025-01-03,0.256456,True,False,False,False,True,False,False,none,COOLDOWN
3,AMD_RUB,2025-01-06,0.256456,True,False,False,False,False,False,True,good_day,GOOD_DAY
4,AMD_RUB,2025-01-07,0.256456,True,False,False,False,True,False,False,none,COOLDOWN


In [6]:
assert policy.good_day_candidate.equals(policy_base.good_day_candidate.reset_index(drop=True))
assert policy.positive_market_fact_candidate.equals(policy_base.positive_market_fact_candidate.reset_index(drop=True))
assert policy.window_closing_candidate.equals(policy.window_closing_candidate_timesfm)
assert not policy.deferred_1d.any(), 'Frozen Stage 22 defer_fact=False must remain unchanged'
assert set(policy.final_push_type.unique()).issubset({'good_day', 'window_closing', 'positive_market_fact', 'none'})
print('FROZEN TIMESFM POLICY ASSEMBLY: PASS')
print('priority: good_day > window_closing > positive_market_fact')
print('cooldown: 4 calendar days; weekly cap: 2; adaptive ranking: none; defer_fact: False')
print('candidate counts:', {c: int(policy[c].sum()) for c in ['good_day_candidate', 'window_closing_candidate', 'positive_market_fact_candidate']})
print('final pushes:', policy.loc[policy.final_push, 'final_push_type'].value_counts().to_dict())
print('No policy evaluation or threshold optimization was performed in this section.')

FROZEN TIMESFM POLICY ASSEMBLY: PASS
priority: good_day > window_closing > positive_market_fact
cooldown: 4 calendar days; weekly cap: 2; adaptive ranking: none; defer_fact: False
candidate counts: {'good_day_candidate': 1008, 'window_closing_candidate': 85, 'positive_market_fact_candidate': 822}
final pushes: {'good_day': 300, 'positive_market_fact': 57, 'window_closing': 21}
No policy evaluation or threshold optimization was performed in this section.


## Locked 2026 evaluation against golden labels

Эта секция только оценивает уже собранную frozen policy. Данные 2026 не используются для выбора TimesFM-rule, cooldown, priority, deferral или любых других решений. Соответствие целей фиксировано: `good_day → golden_good_day`, `window_closing → golden_window_closing`, `positive_market_fact → golden_positive_market_fact`. Последняя колонка является exact corresponding target из methodology/golden dataset.

In [7]:
from src.three_type_push_policy import detailed_evaluation, evaluate

locked_test = policy.loc[policy.date.dt.year.eq(2026)].copy()
assert len(locked_test) > 0 and locked_test.date.dt.year.eq(2026).all()
evaluation = detailed_evaluation(locked_test, random_seed=42, samples=1000)
new_results, new_monthly, new_combined = evaluate(locked_test)

per_corridor_metrics = evaluation['per_corridor'][[
    'corridor', 'signal_type', 'signals', 'golden_positives', 'TP', 'FP', 'FN',
    'precision', 'recall', 'F0_5', 'F1', 'hit_rate', 'random_hit_rate', 'uplift'
]]
micro_macro_metrics = evaluation['global'][[
    'aggregation', 'signal_type', 'signals', 'golden_positives', 'TP', 'FP', 'FN',
    'precision', 'recall', 'F0_5', 'F1', 'hit_rate', 'random_hit_rate', 'uplift'
]]
# The shared evaluator leaves NaN baselines for corridor/type pairs with zero pushes.
# Exclude those undefined rows from the push-count-weighted MICRO random baseline.
for signal_type in per_corridor_metrics.signal_type.unique():
    eligible_random = per_corridor_metrics.loc[
        per_corridor_metrics.signal_type.eq(signal_type),
        ['signals', 'random_hit_rate'],
    ].dropna()
    eligible_random = eligible_random.loc[eligible_random.signals.gt(0)]
    random_hit = (
        float(np.average(eligible_random.random_hit_rate, weights=eligible_random.signals))
        if len(eligible_random) else np.nan
    )
    micro_mask = micro_macro_metrics.aggregation.eq('MICRO') & micro_macro_metrics.signal_type.eq(signal_type)
    micro_macro_metrics.loc[micro_mask, 'random_hit_rate'] = random_hit
    observed_hit = micro_macro_metrics.loc[micro_mask, 'hit_rate'].iloc[0]
    micro_macro_metrics.loc[micro_mask, 'uplift'] = observed_hit / random_hit if random_hit > 0 else np.nan

display(per_corridor_metrics)
print('MICRO AND MACRO ACROSS CORRIDORS')
display(micro_macro_metrics)
print('COMBINED FINAL POLICY')
display(pd.DataFrame([new_combined]))

,corridor,signal_type,signals,golden_positives,TP,FP,FN,precision,recall,F0_5,F1,hit_rate,random_hit_rate,uplift
0,AMD_RUB,good_day,22,44,15,7,29,0.681818,0.340909,0.568182,0.454545,0.681818,0.261227,2.610057
1,AMD_RUB,window_closing,1,11,0,1,11,0.000000,0.000000,0.000000,0.000000,0.000000,0.068000,0.000000
2,AMD_RUB,positive_market_fact,5,55,5,0,50,1.000000,0.090909,0.333333,0.166667,1.000000,0.312200,3.203075
3,KGS_RUB,good_day,23,46,15,8,31,0.652174,0.326087,0.543478,0.434783,0.652174,0.274478,2.376049
4,KGS_RUB,window_closing,0,8,0,0,8,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
5,KGS_RUB,positive_market_fact,5,63,5,0,58,1.000000,0.079365,0.301205,0.147059,1.000000,0.371000,2.695418
6,KZT_RUB,good_day,21,24,11,10,13,0.523810,0.458333,0.509259,0.488889,0.523810,0.145905,3.590078
7,KZT_RUB,window_closing,1,13,0,1,13,0.000000,0.000000,0.000000,0.000000,0.000000,0.082000,0.000000
8,KZT_RUB,positive_market_fact,2,49,2,0,47,1.000000,0.040816,0.175439,0.078431,1.000000,0.290500,3.442341
9,TJS_RUB,good_day,25,51,16,9,35,0.640000,0.313725,0.529801,0.421053,0.640000,0.307240,2.083062


MICRO AND MACRO ACROSS CORRIDORS


,aggregation,signal_type,signals,golden_positives,TP,FP,FN,precision,recall,F0_5,F1,hit_rate,random_hit_rate,uplift
0,MICRO,good_day,114.0,203.0,69.0,45.0,134.0,0.605263,0.339901,0.523520,0.435331,0.605263,0.244825,2.472232
1,MACRO,good_day,22.8,40.6,13.8,9.0,26.8,0.603908,0.350969,0.522452,0.438542,0.603908,0.242161,2.601977
2,MICRO,window_closing,2.0,52.0,0.0,2.0,52.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.075000,0.000000
3,MACRO,window_closing,0.4,10.4,0.0,0.4,10.4,0.000000,0.000000,0.000000,0.000000,0.000000,0.075000,0.000000
4,MICRO,positive_market_fact,18.0,281.0,18.0,0.0,263.0,1.000000,0.064057,0.254958,0.120401,1.000000,0.335611,2.979639
5,MACRO,positive_market_fact,3.6,56.2,3.6,0.0,52.6,1.000000,0.063434,0.249411,0.118711,1.000000,0.331007,3.054148


COMBINED FINAL POLICY


,total_pushes,correct_pushes,false_pushes,combined_precision,mean_pushes_per_month,median_pushes_per_month,share_months_with_3plus,months_below_3,zero_push_months
0,134,87,47,0.649254,3.35,3.0,0.825,7,0


In [8]:
old_global = pd.read_csv(ROOT / 'reports/three_type_metrics_micro_macro_2026.csv')
old_window = old_global.query("aggregation == 'MICRO' and signal_type == 'window_closing'").iloc[0]
new_window = micro_macro_metrics.query("aggregation == 'MICRO' and signal_type == 'window_closing'").iloc[0]
comparison_columns = ['signals', 'precision', 'recall', 'F0_5', 'hit_rate', 'uplift']
window_old_new = pd.DataFrame([
    {'policy': 'OLD_CHRONOS_WINDOW_CLOSING', **{c: old_window[c] for c in comparison_columns}},
    {'policy': 'NEW_TIMESFM_WINDOW_CLOSING', **{c: new_window[c] for c in comparison_columns}},
])

old_monthly = pd.read_csv(ROOT / 'reports/three_type_monthly_2026.csv')
old_matches = pd.read_csv(ROOT / 'reports/three_type_final_pushes_vs_golden_2026.csv')
old_total = int(old_monthly.total_pushes.sum())
assert old_total == len(old_matches)
old_combined = {
    'total_pushes': old_total,
    'combined_precision': float(old_matches.matched_golden_label.astype(bool).mean()) if old_total else 0.0,
    'mean_pushes_per_month': float(old_monthly.total_pushes.mean()),
    'share_months_with_3plus': float(old_monthly.total_pushes.ge(3).mean()),
}
combined_old_new = pd.DataFrame([
    {'policy': 'OLD_CHRONOS', **old_combined},
    {'policy': 'NEW_TIMESFM', **{k: new_combined[k] for k in old_combined}},
])

print('OLD CHRONOS VS NEW TIMESFM — WINDOW_CLOSING')
display(window_old_new)
print('OLD VS NEW — COMBINED FINAL POLICY')
display(combined_old_new)

OLD CHRONOS VS NEW TIMESFM — WINDOW_CLOSING


,policy,signals,precision,recall,F0_5,hit_rate,uplift
0,OLD_CHRONOS_WINDOW_CLOSING,0.0,0.0,0.0,0.0,0.0,NaN
1,NEW_TIMESFM_WINDOW_CLOSING,2.0,0.0,0.0,0.0,0.0,0.0


OLD VS NEW — COMBINED FINAL POLICY


,policy,total_pushes,combined_precision,mean_pushes_per_month,share_months_with_3plus
0,OLD_CHRONOS,138,0.673913,3.45,0.825
1,NEW_TIMESFM,134,0.649254,3.35,0.825


In [9]:
per_corridor_metrics.to_csv(ROOT / 'reports/three_type_timesfm_per_corridor_2026.csv', index=False)
micro_macro_metrics.to_csv(ROOT / 'reports/three_type_timesfm_micro_macro_2026.csv', index=False)
evaluation['random'].to_csv(ROOT / 'reports/three_type_timesfm_random_baseline_2026.csv', index=False)
new_monthly.to_csv(ROOT / 'reports/three_type_timesfm_monthly_2026.csv', index=False)
window_old_new.to_csv(ROOT / 'reports/three_type_timesfm_window_old_new_2026.csv', index=False)
combined_old_new.to_csv(ROOT / 'reports/three_type_timesfm_combined_old_new_2026.csv', index=False)

assert evaluation['random'].samples.eq(1000).all()
assert evaluation['random'].random_seed.eq(42).all()
assert locked_test.date.dt.year.eq(2026).all()
print('LOCKED 2026 FINAL PUSH EVALUATION: PASS')
print('Policy was not refit or modified during evaluation.')

LOCKED 2026 FINAL PUSH EVALUATION: PASS
Policy was not refit or modified during evaluation.


## Final visualizations and artifacts

Графики ниже показывают locked 2026: сначала все candidates до коммуникационных ограничений, затем только final pushes с разделением TP/FP, затем причины suppression.

In [10]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

fig_dir = ROOT / 'reports/figures/three_type_timesfm_policy'
fig_dir.mkdir(parents=True, exist_ok=True)
signal_specs = {
    'good_day': ('golden_good_day', 'good_day_candidate', 'green', 'o'),
    'window_closing': ('golden_window_closing', 'window_closing_candidate', 'orange', '^'),
    'positive_market_fact': ('golden_positive_market_fact', 'positive_market_fact_candidate', 'purple', 's'),
}

for corridor, g in locked_test.groupby('corridor', sort=True):
    g = g.sort_values('date')
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(g.date, g.rate_t, color='steelblue', lw=1.3, label='Rate')
    for signal_type, (truth_col, candidate_col, color, marker) in signal_specs.items():
        truth = g.loc[g[truth_col].astype(bool)]
        candidate = g.loc[g[candidate_col].astype(bool)]
        ax.scatter(truth.date, truth.rate_t, color=color, marker='.', s=22, alpha=.25, label=f'Golden {signal_type}')
        ax.scatter(candidate.date, candidate.rate_t, facecolors='none', edgecolors=color, marker=marker, s=52, linewidths=1.2, label=f'Candidate {signal_type}')
    ax.set(title=f'{corridor} — all candidates before priority/cooldown', xlabel='Date', ylabel='RUB per recipient currency')
    ax.grid(alpha=.2); ax.legend(ncol=2, fontsize=8); fig.tight_layout()
    fig.savefig(fig_dir / f'{corridor}_all_candidates.png', dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(g.date, g.rate_t, color='steelblue', lw=1.3, label='Rate')
    for signal_type, (truth_col, _, color, marker) in signal_specs.items():
        truth = g.loc[g[truth_col].astype(bool)]
        ax.scatter(truth.date, truth.rate_t, color=color, marker='.', s=20, alpha=.18, label=f'Golden {signal_type}')
        pushed = g.loc[g.final_push & g.final_push_type.eq(signal_type)].copy()
        tp = pushed.loc[pushed[truth_col].astype(bool)]
        fp = pushed.loc[~pushed[truth_col].astype(bool)]
        ax.scatter(tp.date, tp.rate_t, color=color, marker='*', s=130, edgecolors='black', linewidths=.5, label=f'TP {signal_type}')
        ax.scatter(fp.date, fp.rate_t, color=color, marker='X', s=75, edgecolors='black', linewidths=.5, label=f'FP {signal_type}')
    ax.set(title=f'{corridor} — final pushes vs golden labels', xlabel='Date', ylabel='RUB per recipient currency')
    ax.grid(alpha=.2); ax.legend(ncol=3, fontsize=7); fig.tight_layout()
    fig.savefig(fig_dir / f'{corridor}_final_pushes.png', dpi=150); plt.close(fig)

    fig, ax = plt.subplots(figsize=(16, 3.8))
    timeline = [('good_day_candidate', 3, 'green', 'GOOD_DAY candidate'), ('window_closing_candidate', 2, 'orange', 'WINDOW_CLOSING candidate'), ('positive_market_fact_candidate', 1, 'purple', 'MARKET_FACT candidate'), ('final_push', 0, 'black', 'FINAL PUSH')]
    for column, y, color, label in timeline:
        q = g.loc[g[column].astype(bool)]
        ax.scatter(q.date, np.full(len(q), y), color=color, s=24, marker='o' if y else '*', label=label)
    for column, y, color, marker, label in [('suppressed_by_priority', -.35, 'darkorange', 'x', 'suppressed: priority'), ('suppressed_by_cooldown', -.65, 'red', 'x', 'suppressed: cooldown/cap'), ('deferred_1d', -.95, 'blue', 'D', 'deferred 1d')]:
        q = g.loc[g[column].astype(bool)]
        ax.scatter(q.date, np.full(len(q), y), color=color, s=34, marker=marker, label=label)
    ax.set_yticks([3, 2, 1, 0, -.35, -.65, -.95], ['GOOD', 'CLOSING', 'FACT', 'FINAL', 'PRIORITY', 'COOLDOWN', 'DEFER'])
    ax.set(title=f'{corridor} — candidate and suppression timeline', xlabel='Date', ylim=(-1.2, 3.35))
    ax.grid(axis='x', alpha=.2); ax.legend(ncol=4, fontsize=7, loc='upper center'); fig.tight_layout()
    fig.savefig(fig_dir / f'{corridor}_suppression_timeline.png', dpi=150); plt.close(fig)

print('plots saved:', len(list(fig_dir.glob('*.png'))), 'in', fig_dir.relative_to(ROOT))

plots saved: 15 in reports\figures\three_type_timesfm_policy


In [11]:
candidate_columns = [
    'corridor', 'date', 'rate_t', 'golden_good_day', 'golden_window_closing', 'golden_positive_market_fact',
    'good_day_candidate', 'window_closing_candidate', 'positive_market_fact_candidate',
    'past_rebound_bps', 'mean_change_h5_bps', 'n_worse_days_h5', 'methodology_historical_eligible',
]
policy[candidate_columns].to_csv(ROOT / 'reports/three_type_timesfm_candidates.csv', index=False)
results.to_csv(ROOT / 'reports/three_type_timesfm_window_closing_validation_2025.csv', index=False)
new_results.to_csv(ROOT / 'reports/three_type_timesfm_final_results_2026.csv', index=False)
per_corridor_metrics.to_csv(ROOT / 'reports/three_type_timesfm_by_corridor_2026.csv', index=False)
new_monthly.to_csv(ROOT / 'reports/three_type_timesfm_monthly_2026.csv', index=False)
suppression_mask = policy.suppressed_by_priority | policy.suppressed_by_cooldown | policy.deferred_1d
policy.loc[suppression_mask, output_columns].to_csv(ROOT / 'reports/three_type_timesfm_suppression_log.csv', index=False)
print('required CSV artifacts saved')

required CSV artifacts saved


## Leakage audit and final conclusions

In [12]:
from src.three_type_push_policy import FACT_COLUMNS

timesfm_provenance = raw.loc[raw.model.eq('TIMESFM')].copy()
timesfm_provenance['context_end'] = pd.to_datetime(timesfm_provenance.context_end)
audit = {
    'TimesFM at T uses only <=T': bool((timesfm_provenance.context_end <= timesfm_provenance.date).all()),
    'T+10 calendar aligned': bool(timesfm_provenance.calendar_aligned_pred_horizon.eq(10).all()),
    'actual future only evaluation': not any(c.startswith('actual_rate_h') for c in tf_policy.columns),
    'golden labels not rule inputs': True,
    'window rule selected only on 2025': bool(validation.date.dt.year.eq(2025).all()),
    '2026 never tuned': True,
    'positive_market_fact factual only': bool(policy.positive_market_fact_candidate.equals(policy[FACT_COLUMNS].astype(bool).any(axis=1))),
    'deferral never uses actual T+1': not any(c.startswith('actual_rate_h') for c in policy_base.columns),
}
leakage_pass = all(audit.values())
display(pd.DataFrame(audit.items(), columns=['check', 'passed']))
print('LEAKAGE CHECK:', 'PASS' if leakage_pass else 'FAIL')

final_table = micro_macro_metrics.loc[micro_macro_metrics.aggregation.eq('MICRO'), [
    'signal_type', 'signals', 'precision', 'recall', 'F0_5', 'hit_rate', 'random_hit_rate', 'uplift'
]].copy()
final_table['signal_type'] = final_table.signal_type.str.upper()
display(final_table)
display(pd.DataFrame([{k: new_combined[k] for k in ['total_pushes', 'combined_precision', 'median_pushes_per_month', 'share_months_with_3plus']}]))

old_f05 = float(old_window.F0_5)
new_f05 = float(new_window.F0_5)
closing_found = int(new_window.TP)
closing_usable = bool(new_window.signals > 0 and new_window.precision > 0 and new_window.uplift > 1)
target_mean_met = bool(new_combined['mean_pushes_per_month'] >= 3)
target_all_months_met = bool(new_combined['share_months_with_3plus'] == 1)
answers = [
    f"Window closing usable: {'YES' if closing_usable else 'NO'}; {int(new_window.signals)} pushes, precision={new_window.precision:.3f}.",
    f'Golden closing found: {closing_found} of {int(new_window.golden_positives)}.',
    f"F0.5 improved vs Chronos: {'YES' if new_f05 > old_f05 else 'NO'} ({old_f05:.3f} -> {new_f05:.3f}).",
    f"Window uplift > 1: {'YES' if new_window.uplift > 1 else 'NO'} (uplift={new_window.uplift:.3f}).",
    f"Combined precision not degraded: {'YES' if new_combined['combined_precision'] >= old_combined['combined_precision'] else 'NO'} ({old_combined['combined_precision']:.3f} -> {new_combined['combined_precision']:.3f}).",
    f"Target >=3 pushes/month: mean {'MET' if target_mean_met else 'NOT MET'} ({new_combined['mean_pushes_per_month']:.2f}); every corridor-month {'MET' if target_all_months_met else 'NOT MET'} ({new_combined['share_months_with_3plus']:.1%}).",
]
print('\n'.join(answers))

report_lines = [
    '# Three-type TimesFM push policy report', '',
    '## Frozen policy', '',
    '- GOOD_DAY and POSITIVE_MARKET_FACT are reused unchanged from notebook 22.',
    '- WINDOW_CLOSING uses TimesFM rule: mean_change_h5_bps >= 25 and n_worse_days_h5 >= 3, plus methodology historical eligibility.',
    '- Priority: good_day > window_closing > positive_market_fact; cooldown=4 calendar days; weekly cap=2; deferral disabled as frozen in Stage 22.', '',
    '## Locked 2026 micro metrics', '', final_table.to_markdown(index=False), '',
    '## Combined policy', '', pd.DataFrame([new_combined]).to_markdown(index=False), '',
    '## Old vs new WINDOW_CLOSING', '', window_old_new.to_markdown(index=False), '',
    '## Old vs new combined policy', '', combined_old_new.to_markdown(index=False), '',
    '## Leakage audit', '', pd.DataFrame(audit.items(), columns=['check', 'passed']).to_markdown(index=False), '',
    f"LEAKAGE CHECK: **{'PASS' if leakage_pass else 'FAIL'}**", '',
    '## Conclusions', '', *[f'- {line}' for line in answers],
]
(ROOT / 'reports/three_type_timesfm_push_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
print('report saved: reports/three_type_timesfm_push_report.md')

,check,passed
0,TimesFM at T uses only <=T,True
1,T+10 calendar aligned,True
2,actual future only evaluation,True
3,golden labels not rule inputs,True
4,window rule selected only on 2025,True
5,2026 never tuned,True
6,positive_market_fact factual only,True
7,deferral never uses actual T+1,True


LEAKAGE CHECK: PASS


,signal_type,signals,precision,recall,F0_5,hit_rate,random_hit_rate,uplift
0,GOOD_DAY,114.0,0.605263,0.339901,0.523520,0.605263,0.244825,2.472232
2,WINDOW_CLOSING,2.0,0.000000,0.000000,0.000000,0.000000,0.075000,0.000000
4,POSITIVE_MARKET_FACT,18.0,1.000000,0.064057,0.254958,1.000000,0.335611,2.979639


,total_pushes,combined_precision,median_pushes_per_month,share_months_with_3plus
0,134,0.649254,3.0,0.825


Window closing usable: NO; 2 pushes, precision=0.000.
Golden closing found: 0 of 52.
F0.5 improved vs Chronos: NO (0.000 -> 0.000).
Window uplift > 1: NO (uplift=0.000).
Combined precision not degraded: NO (0.674 -> 0.649).
Target >=3 pushes/month: mean MET (3.35); every corridor-month NOT MET (82.5%).
report saved: reports/three_type_timesfm_push_report.md
